In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

ir_report = pd.read_csv('nfl_injuries_full.csv')
weekly_report = pd.read_csv('injury_report.csv')

# Break down the notes columns (new injury column and transaction column)

ir_report['notes_clean'] = ir_report['Notes'].str.lower().str.strip()
weekly_report['notes_clean'] = weekly_report['report_status'].str.lower().str.strip()


# Add new column to identify source of data
weekly_report['source'] = 'weekly_report'
ir_report['source'] = 'ir_transaction'


# Merge acquired and relinquished columns in ir report to one player name column
ir_report['full_name'] = ir_report['Acquired'].combine_first(ir_report['Relinquished'])


# change column names to match for concatenation
ir_report = ir_report.rename(columns={
    'Date':'date',
    'Team':'team'})

# Now that the injury report (weekly) is being added, i can remove the out status from ir_reports
ir_report = ir_report[~ir_report['notes_clean'].str.contains("(out)", regex=False, na=False)]

# Removes (aka) names
ir_report['full_name'] = ir_report['full_name'].str.split('/').str[0]
ir_report['full_name'] = ir_report['full_name'].str.replace(r"\(.*?\)", "", regex=True).str.strip()
ir_report['full_name'] = ir_report['full_name'].str.strip()


# Change date_modified format to match Date format in injuries dataframe
weekly_report['date'] = weekly_report['date_modified']
weekly_report['date'] = pd.to_datetime(
    weekly_report['date'], 
    format="%Y-%m-%dT%H:%M:%SZ"
).dt.date
ir_report['date'] = pd.to_datetime(ir_report['date'])
weekly_report['date'] = pd.to_datetime(weekly_report['date'])



# CONCAT AND SORT
injuries = pd.concat([ir_report, weekly_report], ignore_index=True, sort=False)

injuries = injuries.sort_values(
    by=['date']
).reset_index(drop=True)



# Extract transaction type from notes
def extract_action(note):
    if not isinstance(note, str):
        return None
    if 'ir' in note:
        if 'placed' in note:
            return 'Placed on IR'
        if 'activated' in note:
            return 'Activated from IR'
    elif 'out' in note:
        if 'for season' in note:
            return 'Out for Season'
        elif 'indefinitely' in note:
            return 'Out Indefinitely'
        return 'Out'
    elif 'physically unable to perform' in note:
        if 'activated' in note:
            return 'Activated from PUP'
        elif 'placed' in note:
            return 'Placed on PUP'
    elif 'covid' in note:
        if 'placed' in note:
            return 'Placed on COVID list'
        elif 'activated' in note:
            return 'Activated from COVID list'
    elif 'non-football' in note:
        if 'placed' in note:
            return 'Placed on NFI'
        elif 'activated' in note:
            return 'Activated from NFI'
    elif 'dtd' in note:
        return 'Day-to-Day'
    elif 'doubtful' in note:
        return 'Doubtful'
    elif 'probable' in note:
        return 'Probable'
    elif 'questionable' in note:
        return 'Questionable'
    elif 'note' in note:
        return 'Undetermined Game Status'
    elif 'list' in note:
        return 'Special List Transaction'
    else:
        return 'Other'

#Extract General Injury Location
injury_dict = {
    'Ankle': ['ankle', 'achilles', 'achiles'],
    'Knee': ['knee', 'acl', 'mcl', 'meniscus', 'patellar', 'patella', 'lcl', 'pcl'],
    'Hand/Wrist': ['hand', 'wrist', 'finger', 'thumb'],
    'Shoulder': ['shoulder', 'collarbone', 'clavicle'],
    'Head':['head', 'concussion', 'eye', 'jaw', 'throat', 'nose', 'teeth', 'chin'],
    'Back/Spine': ['back', 'neck', 'spine', 'spinal', 'oblique', 'pelvis'],
    'Foot': [r'\bfoot\b', 'toe', 'heel', 'feet'],
    'Leg': ['hamstring', 'groin', 'calf', 'hip', 'quad', 'thigh', 'tibia', 'fibula', 'leg', 'shin', 'quadricep', 'quadriceps', 'glute',
    'glutes', 'buttocks'],
    'Arm': ['elbow', 'biceps', 'forearm', 'triceps', 'arm', 'tricep', 'bicep'],
    'Ribs': ['ribs', 'rib'],
    'Chest/Abdominal': ['chest', 'abdominal', 'pectoral', 'core', 'abdomen'],
    'Illness': ['illness', 'ill', 'infection', 'migraine'],
    'Upper Body': ['upper body'],
    'Lower Body': ['lower body'],
    'Undisclosed': ['undisclosed', 'various', 'stinger'],
    'Not Injury Related': ['not injury related']
}

def extract_injury(note):
    if not isinstance(note, str):
        return None
    for k,v in injury_dict.items():
        for term in v:
            if term in note:
                return k

    return 'Other'

# Extract Severity of the Injury
severity_terms = {
    'Torn': ['torn'],
    'Ruptured': ['ruptured'],
    'Sprain': ['sprain', 'sprained'],
    'Strain': ['strain', 'strained'],
    'Fracture': ['fracture', 'fractured'],
    'Mild': ['bruise', 'contusion', 'minor'],
    'Dislocated': ['dislocation', 'dislocated', 'dislocate']
}

def extract_severity(note):
    if not isinstance(note, str):
        return None
    for k,v in severity_terms.items():
        for term in v:
            if term in note:
                return k
    else:
        return 'Unknown'
    
# Extract More Detailed Injury Location
detail_dict = {
    'Achilles': ['achilles', 'achiles'],
    'Ankle': ['ankle'],
    'ACL': ['acl'],
    'Knee': ['knee'],
    'MCL': ['mcl'],
    'Meniscus': ['meniscus'],
    'Concussion': ['concussion'],
    'Hamstring': ['hamstring'],
    'Shoulder': ['shoulder'],
    'Foot': [r'\bfoot\b'],
    'Wrist': ['wrist'],
    'Groin': ['groin'],
    'Pectoral': ['pectoral'],
    'Calf': ['calf'],
    'Hip': ['hip'],
    'Quad': ['quad', 'quadricep', 'quadriceps'],
    'Back': ['back'],
    'Neck': ['neck'],
    'Toe': ['toe'],
    'Finger': ['finger'],
    'Ribs': ['ribs', 'rib'],
    'Elbow': ['elbow'],
    'Chest': ['chest'],
    'Abdominal': ['abdominal'],
    'Biceps': ['biceps', 'bicep'],
    'Forearm': ['forearm'],
    'Triceps': ['triceps', 'tricep'],
    'Thigh': ['thigh'],
    'Thumb': ['thumb'],
    'Hand': ['hand'],
    'Collarbone': ['collarbone'],
    'Head': ['head'],
    'Illness': ['illness'],
    'Patellar': ['patellar'],
    'Patella': ['patella'],
    'LCL': ['lcl'],
    'PCL': ['pcl'],
    'Leg': ['leg'],
    'Shin': ['shin'],
    'Tibia': ['tibia'],
    'Fibula': ['fibula'],
    'Eye': ['eye'],
    'Core': ['core'],
    'Heel': ['heel'],
    'Oblique': ['oblique'],
    'Spine': ['spinal', 'spine'],
    'Abdomen': ['abdomen', 'abdominal'],
    'Pectoral': ['pectoral'],
    'Stinger': ['stinger'],
    'Spleen': ['spleen'],
    'Buttocks': ['buttocks'],
    'Glute': ['glute', 'glutes'],
    'Pelvis': ['pelvis'],
    'Jaw': ['jaw'],
    'Feet': ['feet'],
    'Nose': ['nose'],
    'Throat': ['throat'],
    'Infection': ['infection'],
    'Migraine': ['migraine'],
    'Teeth': ['teeth'],
    'Chin': ['chin']
}

def injury_detail(note):
    if not isinstance(note, str):
        return None
    for k,v in detail_dict.items():
        for term in v:
            if term in note:
                return k

    return 'Unknown'

def surgery_mention(note):
    if not isinstance(note, str):
        return None
    if 'surgery' in note:
        return 1
    else:
        return 0
    
# Create the new columns consisting of injury data

mask_ir = injuries['source'] == 'ir_transaction'

injuries.loc[mask_ir, 'action'] = injuries.loc[mask_ir, 'notes_clean'].apply(extract_action)
injuries.loc[mask_ir, 'injury_type'] = injuries.loc[mask_ir, 'notes_clean'].apply(extract_injury)
injuries.loc[mask_ir, 'severity'] = injuries.loc[mask_ir, 'notes_clean'].apply(extract_severity)
injuries.loc[mask_ir, 'injury_detail'] = injuries.loc[mask_ir, 'notes_clean'].apply(injury_detail)
injuries.loc[mask_ir, 'surgery_mention'] = injuries.loc[mask_ir, 'notes_clean'].apply(surgery_mention)


mask_wr = injuries['source'] == 'weekly_report'

injuries.loc[mask_wr, 'action'] = injuries.loc[mask_wr, 'notes_clean'].apply(extract_action)
cols_combined = injuries.loc[mask_wr, 'report_primary_injury'].str.lower().str.strip().fillna(injuries.loc[mask_wr, 'practice_primary_injury']).str.lower().str.strip()
injuries.loc[mask_wr, 'severity'] = injuries.loc[mask_wr, 'notes_clean'].apply(extract_severity)
injuries.loc[mask_wr, 'injury_type'] = cols_combined.apply(extract_injury)
injuries.loc[mask_wr, 'injury_detail'] = cols_combined.apply(injury_detail)

#===========
'''
print(injuries['action'].value_counts())

others = injuries.loc[injuries["action"] == "Surgery Update", "notes_clean"]
print(others.value_counts())
'''
#============
'''
print(injuries['injury_type'].value_counts())

others2 = injuries.loc[injuries["injury_type"] == "Other", 'report_primary_injury']
print(others2.value_counts())
'''
#============
'''
print(injuries['severity'].value_counts())

others3 = injuries.loc[injuries["severity"] == "Torn", "notes_clean"]
print(others3.value_counts())
'''
#============
'''
print(injuries['injury_detail'].value_counts())

others4 = injuries.loc[injuries["injury_detail"] == "Achilles", "notes_clean"]
print(others4.value_counts())
'''
#============
# Get's rid of bullet points in Acquired and Relinquished columns
bullets = r"^[•\u2022\u25CF\u25E6]\s*"

cols = ["Acquired", "Relinquished", "full_name"]

for c in cols:
    injuries[c] = injuries[c].astype(str).str.replace(bullets, "", regex=True).replace("nan", None).str.strip()

# Generalize team format
nfl_teams = {
    "ARI": ["Cardinals"],
    "ATL": ['Falcons'],
    "BAL": ['Ravens'],
    "BUF": ['Bills'],
    "CAR": ['Panthers'],
    "CHI": ['Bears'],
    "CIN": ['Bengals'],
    "CLE": ['Browns'],
    "DAL": ['Cowboys'],
    "DEN": ['Broncos'],
    "DET": ['Lions'],
    "GB": ['Packers'],
    "HOU": ['Texans'],
    "IND": ['Colts'],
    "JAX": ['Jaguars'],
    "KC": ['Chiefs'],
    "LV": ['Raiders'],
    "LAC": ['Chargers'],
    "LAR": ['Rams'],
    "MIA": ['Dolphins'],
    "MIN": ['Vikings'],
    "NE": ['Patriots'],
    "NO": ['Saints'],
    "NYG": ['Giants'],
    "NYJ": ['Jets'],
    "PHI": ['Eagles'],
    "PIT": ['Steelers'],
    "SEA": ['Seahawks'],
    "SF": ['49ers'],
    "TB": ['Buccaneers'],
    "TEN": ['Titans'],
    "WAS": ["Commanders", 'Washington', 'Redskins']
}
def change_team(note):
    if not isinstance(note, str):
        return None
    for k,v in nfl_teams.items():
        for term in v:
            if term in note:
                return k

    return

injuries.loc[mask_ir, 'team'] = injuries.loc[mask_ir, 'team'].apply(change_team)

# Creating a new column to identify and group injuries (InjuryID)
injuries = injuries.sort_values(['full_name', 'injury_detail', 'date']).reset_index(drop=True)


# Add the year for the season column in events from the transaction table
mask_missing = injuries['season'].isnull()
dates_missing = injuries.loc[mask_missing, 'date']

injuries.loc[mask_missing, 'season'] = np.where(
    dates_missing.dt.month <= 2,
    dates_missing.dt.year - 1,
    dates_missing.dt.year
)


# Update the Mask
mask_ir = injuries['source'] == 'ir_transaction'

# Force Convert Season to Integer
injuries.loc[mask_ir, 'season_clean'] = (
    pd.to_numeric(injuries.loc[mask_ir, 'season'], errors='coerce')
    .astype('Int64')
)


season_start_dates = {
    2011: '2011-09-08',
    2012: '2012-09-05',    #WEDNESDAY
    2013: '2013-09-05',
    2014: '2014-09-04',
    2015: '2015-09-10',
    2016: '2016-09-08',
    2017: '2017-09-07',
    2018: '2018-09-06',
    2019: '2019-09-05',
    2020: '2020-09-10',
    2021: '2021-09-09',
    2022: '2022-09-08',
    2023: '2023-09-07',
    2024: '2024-09-05',
    2025: '2025-09-04',

}

start_dates = pd.to_datetime(injuries.loc[mask_ir, 'season_clean'].map(season_start_dates))

# Calculate Weeks
# We use .dt.days to get the number of days, then integer divide by 7
weeks_diff = (pd.to_datetime(injuries.loc[mask_ir, 'date']) - start_dates).dt.days // 7 + 1

# Assign result
injuries.loc[mask_ir, 'week'] = weeks_diff

# Save to CSV

injuries.to_csv('nfl_injuries_cleaned.csv', index=False)


/var/folders/0m/ynxt4vt11jzfr3w6dq2dk_bw0000gn/T/ipykernel_7424/4206035626.py:12: DtypeWarning: Columns (0: season_type) have mixed types. Specify dtype option on import or set low_memory=False.
  weekly_report = pd.read_csv('injury_report.csv')


In [11]:
# IDENTIFY INJURY IDs ===========================================================================================================================================================

# Set the maximum amount of days for it to be considered a different injury, IR limit too
MAX_GAP_DAYS = 30

IR_CONTINUATION_DAYS = 270

# create a previous date column by grouping by person and their injury. Marks the last date that a player recorded to have that same injury
injuries['on_ir'] = injuries['action'].str.contains(
    'Placed on IR', case=False, na=False
)

injuries['prev_on_ir'] = injuries.groupby(
    ['full_name', 'injury_detail']
)['on_ir'].shift()

injuries['prev_season'] = injuries.groupby(
    ['full_name', 'injury_detail']
)['season'].shift()

injuries['prev_date'] = injuries.groupby(
    ['full_name', 'injury_detail']
)['date'].shift()

# Counts the number of days since once person was recorded to last have that same injury
injuries['days_since_prev'] = (injuries['date'] - injuries['prev_date']).dt.days

# if the days between injuries is too great, flag it as a new injury
injuries['new_injury_flag'] = (
    # First occurrence
    injuries['prev_date'].isna() |

    # Long gap with NO IR involved
    (
        (injuries['days_since_prev'] > MAX_GAP_DAYS) &
        ~(injuries['on_ir'] | injuries['prev_on_ir'])
    ) |

    # IR involved but exceeds realistic rehab window
    (
        (injuries['days_since_prev'] > IR_CONTINUATION_DAYS) &
        (injuries['on_ir'] | injuries['prev_on_ir'])
    ) |

    # Hard season break (true reinjury years later)
    (
        (injuries['season'] - injuries['prev_season'] >= 2)
    )
)

# Cumulate injury ids
injuries['injury_id'] = (
    injuries.groupby(['full_name', 'injury_detail'])['new_injury_flag']
    .cumsum()
)

# make the ids unique to the player and the injury
injuries['injury_id'] = (
    injuries['full_name'] + "_" +
    injuries['injury_detail'] + "_" +
    injuries['injury_id'].astype(str)
)

injuries['injury_split_reason'] = np.select(
    [
        injuries['prev_date'].isna(),
        (injuries['days_since_prev'] > MAX_GAP_DAYS) & ~(injuries['on_ir'] | injuries['prev_on_ir']),
        (injuries['days_since_prev'] > IR_CONTINUATION_DAYS),
        (injuries['season'] - injuries['prev_season'] >= 2)
    ],
    [
        'first_occurrence',
        'gap_no_ir',
        'ir_too_long',
        'multi_season_break'
    ],
    default='continuation'
)

#Save to csv
injuries.to_csv('nfl_injuries_cleaned.csv', index=False)

In [12]:
import pandas as pd
import numpy as np

# COLUMN ENGINEERING
new_order = ['injury_id', 'date', 'team', 'full_name', 'season', 'week', 'game_type', 'action', 'injury_type', 'severity', 'injury_detail', 'surgery_mention', 'position', 'source', 'season_clean', 'prev_date', 'days_since_prev', 'new_injury_flag', 'on_ir', 'prev_on_ir', 'prev_season', 'injury_split_reason', 'Acquired', 'Relinquished', 'Notes', 'notes_clean', 'first_name', 'last_name', 'gsis_id', 'report_primary_injury', 'report_secondary_injury', 'report_status', 'practice_primary_injury', 'practice_secondary_injury', 'practice_status', 'date_modified']

injuries = injuries[new_order]
injuries = injuries.drop(columns=['notes_clean', 'Acquired', 'Relinquished'])
injuries = injuries.rename(columns={'Notes': 'notes'})

injuries.to_csv('nfl_injuries_cleaned.csv')

In [13]:
distinct_teams = (injuries['injury_id'].unique())
print(len(distinct_teams))

51280


In [14]:
# EXTRACTING "Injured during the play" sentences from PBP information
import pandas as pd

pbp = pd.read_csv('pbp.csv')

/var/folders/0m/ynxt4vt11jzfr3w6dq2dk_bw0000gn/T/ipykernel_52922/1243204716.py:4: DtypeWarning: Columns (45,179,180,182,183,189,190,193,194,197,198,203,204,205,206,209,210,213,214,218,219,220,222,224,226,233,234,235,236,237,238,243,244,245,248,249,253,254,255,260,262,263,266,267,268,269,283,284,302,332) have mixed types. Specify dtype option on import or set low_memory=False.
  pbp = pd.read_csv('pbp.csv')


In [18]:
pbp_order = ['play_id','game_id','old_game_id','home_team','away_team','season_type','week','yardline_100','game_date','qtr','desc','play_type',
'down','goal_to_go','time','yrdln','ydstogo','ydsnet','epa','season','start_time','stadium','weather','nfl_api_id','away_score','home_score',
'location','spread_line','total_line','div_game','roof','surface','temp','wind','home_coach','away_coach','stadium_id','game_stadium']

pbp = pbp[pbp_order]
pbp.to_csv('pbp_cleaned.csv')

In [ ]:
import pandas as pd
import re

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)



pattern = r'([A-Z]{2,3}-\d{1,3}-(?:(?!\.\s).)*?)(?=\s+was injured)'

pbp['injured_players'] = pbp['desc'].apply(lambda x: re.findall(pattern, x))

max = 0
for row in pbp['injured_players']:
    if len(row) > max:
        max = len(row)

injury_table = pbp.explode('injured_players').dropna(subset=['injured_players'])
split_data = injury_table['injured_players'].str.split('-', n=2, expand=True)
split_data.columns = ['team', 'number', 'player']
final_injury_report = pd.concat([injury_table, split_data], axis=1)

final_injury_report = final_injury_report[['game_id', 'play_id', 'game_date', 'team', 'number', 'player']]

pbp_injury_mentions.to_csv('pbp_injury_mentions.csv', index=False)

print(len(final_injury_report)) # logged injuries
print(len(injuries))
print(len(pbp))


12268
98522
723252


In [ ]:
import pandas as pd
import re
import numpy as np

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

# ===============================
# LOAD DATA
report = pd.read_csv('nfl_injuries_cleaned.csv')
logged = pd.read_csv('inGameInjuries.csv')
pbp = pd.read_csv('pbp_cleaned.csv')

# ===============================
# LOGGED NAME/TEAM COLUMNS
# ===============================
logged_name_col = 'player_name' if 'player_name' in logged.columns else 'player'
logged_team_col = 'player_team' if 'player_team' in logged.columns else 'team'

# ===============================
# DROP ROWS WITH NULL NAMES (can't match without a name)
report = report.dropna(subset=['full_name'])

# ===============================
# STANDARDIZE DATES
report['date'] = pd.to_datetime(report['date'], errors='coerce')
logged['game_date'] = pd.to_datetime(logged['game_date'], errors='coerce')
pbp['game_date'] = pd.to_datetime(pbp['game_date'], errors='coerce')

# Drop bad dates early
report = report.dropna(subset=['date'])
logged = logged.dropna(subset=['game_date'])
pbp = pbp.dropna(subset=['game_date'])

# ===============================
# NORMALIZE TEAM
report['team'] = report['team'].str.upper().str.strip()
if 'player_team' in logged.columns:
    logged['player_team'] = logged['player_team'].astype(str).str.upper().str.strip().replace('NAN', None)
if 'team' not in logged.columns:
    logged['team'] = logged.get('player_team')
else:
    logged['team'] = logged['team'].fillna(logged.get('player_team'))
logged['team'] = logged['team'].astype(str).str.upper().str.strip().replace('NAN', None)
pbp['home_team'] = pbp['home_team'].str.upper().str.strip()
pbp['away_team'] = pbp['away_team'].str.upper().str.strip()

# Map historical/alternate team abbreviations to one canonical code
TEAM_ALIASES = {
    'OAK': 'LV',
    'LA': 'LAR',
    'STL': 'LAR',
    'SD': 'LAC',
    'JAC': 'JAX',
}

def canonical_team(value):
    if pd.isna(value):
        return None
    t = str(value).upper().strip()
    if not t or t == 'NAN':
        return None
    return TEAM_ALIASES.get(t, t)

report['team'] = report['team'].apply(canonical_team)
logged['team'] = logged['team'].apply(canonical_team)
if 'player_team' in logged.columns:
    logged['player_team'] = logged['player_team'].apply(canonical_team)
pbp['home_team'] = pbp['home_team'].apply(canonical_team)
pbp['away_team'] = pbp['away_team'].apply(canonical_team)

# If team is missing in logged data, use home/away teams as fallback candidates
if {'home_team', 'away_team'}.issubset(logged.columns):
    logged['home_team'] = logged['home_team'].apply(canonical_team)
    logged['away_team'] = logged['away_team'].apply(canonical_team)

    missing_team_mask = logged['team'].isna() | (logged['team'].astype(str).str.strip() == '')
    if missing_team_mask.any():
        logged_missing_team = logged.loc[missing_team_mask].copy()
        logged_with_team = logged.loc[~missing_team_mask].copy()

        home_fallback = logged_missing_team.copy()
        home_fallback['team'] = home_fallback['home_team']
        home_fallback['team_source'] = 'home_team_fallback'

        away_fallback = logged_missing_team.copy()
        away_fallback['team'] = away_fallback['away_team']
        away_fallback['team_source'] = 'away_team_fallback'

        logged = pd.concat([logged_with_team, home_fallback, away_fallback], ignore_index=True)
        logged = logged[logged['team'].notna()].copy()
        logged = logged.drop_duplicates(subset=['game_id', 'play_id', logged_name_col, 'team'])

        print(f"Filled missing logged team via home/away fallback for {missing_team_mask.sum()} rows")
else:
    print("home_team/away_team not present in logged dataset; team fallback skipped")

# ===============================
# VECTORIZE NAME PARSING

# Define suffixes to remove
SUFFIXES = {'JR.', 'JR', 'SR.', 'SR', 'II', 'III', 'IV', 'V'}

# Cleaning the long names: Harrison Williams Jr. --> H. Williams
def normalize_full_name(name):
    if pd.isna(name):
        return None, None
    parts = str(name).strip().split()
    if not parts:
        return None, None

    if parts[-1].upper().replace('.', '') in SUFFIXES:
        parts = parts[:-1]

    if not parts:
        return None, None

    first_initial = parts[0][0].upper() if parts[0] else None
    last_name = ' '.join(parts[1:]).upper().strip()

    return first_initial, (last_name if last_name else None)

report[['first_initial', 'last_name']] = (
    report['full_name']
    .apply(normalize_full_name)
    .apply(pd.Series)
)

# Cleaning mixed logged formats:
# H.Williams | H.St. Williams | Harrison Williams | Williams | Ha. Williams
def normalize_logged_name(name):
    if pd.isna(name):
        return None, None

    value = str(name).strip()
    if not value:
        return None, None

    value = re.sub(r'^[#\d\-\s]+', '', value)
    value = re.sub(r'\(.*?\)', '', value)
    value = re.sub(r'\s+', ' ', value).strip()
    if not value:
        return None, None

    parts = [p for p in value.split(' ') if p]

    def clean_piece(piece):
        piece = piece.replace('.', '').strip()
        piece = re.sub(r"[^A-Za-z'\-]", '', piece)
        return piece

    if len(parts) == 1:
        raw_token = parts[0].strip()
        if '.' in raw_token:
            chunks = [clean_piece(c) for c in raw_token.split('.') if clean_piece(c)]
            if len(chunks) >= 2:
                first_initial = chunks[0][0].upper() if chunks[0] else None
                last_name = ' '.join(chunks[1:]).upper().strip()
                if last_name:
                    return first_initial, last_name

        token = clean_piece(raw_token)
        if token and len(token) >= 2:
            return None, token.upper()
        return None, None

    first_token = clean_piece(parts[0])
    first_initial = first_token[0].upper() if first_token else None
    last_name = ' '.join(clean_piece(p) for p in parts[1:]).strip().upper()

    if not last_name:
        fallback = clean_piece(parts[-1]) if parts else None
        if fallback and len(fallback) >= 2:
            return first_initial, fallback.upper()
        return None, None

    return first_initial, last_name

logged[['first_initial', 'last_name']] = (
    logged[logged_name_col]
    .apply(normalize_logged_name)
    .apply(pd.Series)
)

# Validate
assert report['first_initial'].notna().all(), "Some report names failed to parse"
logged_unparsed_mask = logged['last_name'].isna() | (logged['last_name'].astype(str).str.strip() == '')
if logged_unparsed_mask.any():
    print(f"Dropping {logged_unparsed_mask.sum()} logged rows with unparsable names")
    logged = logged.loc[~logged_unparsed_mask].copy()

# Spot check for edge cases
report[report['last_name'].str.contains(r'ST\.|VAN|DE |MC ', regex=True)][['full_name', 'last_name']].head(10)

# ===============================
# BUILD REPORT MATCH POOL (ALL REPORT DATES)
# ===============================
report_match_pool = (
    report[[
        'injury_id', 'date', 'first_initial', 'last_name', 'team',
        'full_name', 'injury_detail', 'action', 'injury_type', 'severity'
    ]]
    .dropna(subset=['date', 'last_name', 'team'])
    .drop_duplicates()
)

# ===============================
# BUILD NEXT GAME WINDOW FROM PBP (CRITICAL)
# ===============================
pbp_games = (
    pbp[['game_id', 'game_date', 'home_team', 'away_team']]
    .drop_duplicates()
    .sort_values('game_date')
)

home_games = pbp_games[['game_date', 'home_team']].copy()
home_games.columns = ['game_date', 'team']

away_games = pbp_games[['game_date', 'away_team']].copy()
away_games.columns = ['game_date', 'team']

all_games = pd.concat([home_games, away_games], ignore_index=True)
all_games = all_games.sort_values(['team', 'game_date']).reset_index(drop=True)


def get_next_game_date(row):
    """Find the next game date for the team after the injury date"""
    team = row['team']
    injury_date = row['game_date']

    team_games = all_games[all_games['team'] == team]
    next_games = team_games[team_games['game_date'] > injury_date]

    if len(next_games) > 0:
        return next_games.iloc[0]['game_date']
    else:
        return None


logged['next_game_date'] = logged.apply(get_next_game_date, axis=1)

# Drop rows where we couldn't find a next game
logged_with_window = logged.dropna(subset=['next_game_date'])

print(f"Logged injuries with valid next game: {len(logged_with_window)} / {len(logged)}")

# ===============================
# MERGE ON NAME + TEAM (WITH INITIAL FALLBACK)
# ===============================
logged_with_initial = logged_with_window[logged_with_window['first_initial'].notna()].copy()
logged_without_initial = logged_with_window[logged_with_window['first_initial'].isna()].copy()

candidates_strict = logged_with_initial.merge(
    report_match_pool,
    on=['first_initial', 'last_name', 'team'],
    how='left',
    suffixes=('_log', '_inj')
)
candidates_strict['match_tier'] = 0

candidates_fallback = logged_without_initial.merge(
    report_match_pool,
    on=['last_name', 'team'],
    how='left',
    suffixes=('_log', '_inj')
)
candidates_fallback['match_tier'] = 1

candidates = pd.concat([candidates_strict, candidates_fallback], ignore_index=True)
print(f"Strict merge rows: {len(candidates_strict)} | with report date: {candidates_strict['date'].notna().sum()}")
print(f"Fallback merge rows: {len(candidates_fallback)} | with report date: {candidates_fallback['date'].notna().sum()}")
print(f"Total candidate rows before date window: {len(candidates)}")

# ===============================
# DATE FILTER: Report must appear between injury date and next game
# ===============================
has_report_date = candidates['date'].notna()
after_game_mask = has_report_date & (candidates['date'] >= candidates['game_date'])
before_next_game_mask = has_report_date & (candidates['date'] <= candidates['next_game_date'])
date_window_mask = after_game_mask & before_next_game_mask

print(f"Candidates with report date: {has_report_date.sum()}")
print(f"Dropped (missing report date): {len(candidates) - has_report_date.sum()}")
print(f"Dropped (report before game date): {(has_report_date & ~after_game_mask).sum()}")
print(f"Dropped (report after next game date): {(has_report_date & ~before_next_game_mask).sum()}")

candidates = candidates[date_window_mask]

print(f"Candidates after date filtering: {len(candidates)}")

# ===============================
# SCORE MATCH QUALITY
# ===============================
candidates['date_diff'] = (
    candidates['date'] - candidates['game_date']
).dt.days

# Prioritize IR-confirmed injuries (placed on IR before next game = serious)
candidates['ir_weight'] = (candidates['action'] == 'Placed on IR').astype(int)

# ===============================
# PICK BEST MATCH PER LOGGED EVENT
# ===============================
best_matches = (
    candidates
    .sort_values(['match_tier', 'date_diff', 'ir_weight'], ascending=[True, True, False])
    .groupby(['game_id', 'play_id', logged_name_col, 'team'], as_index=False)
    .first()
)

# ===============================
# OUTPUT: KEEP ALL FIELDS, ADD IDs ONLY
# ===============================
logged.to_csv('F_logged_injuries_with_id.csv', index=False)

# Build confidence for each logged match
best_matches['confidence_level'] = np.select(
    [
        (best_matches['match_tier'] == 0) & (best_matches['date_diff'] <= 3),
        (best_matches['match_tier'] == 0) & (best_matches['date_diff'] <= 7),
        (best_matches['match_tier'] == 1) & (best_matches['date_diff'] <= 3),
    ],
    ['high', 'medium', 'medium'],
    default='low'
)
best_matches['confidence_score'] = best_matches['confidence_level'].map({
    'high': 0.90,
    'medium': 0.70,
    'low': 0.50,
})

# Keep one best logged match per injury_id
best_by_injury = (
    best_matches
    .sort_values(['injury_id', 'match_tier', 'date_diff', 'ir_weight'], ascending=[True, True, True, False])
    .groupby('injury_id', as_index=False)
    .first()
)

# Build Injury Events table (one row per unique injury_id)
injury_event_base = (
    report.sort_values('date')
    .groupby('injury_id', as_index=False)
    .agg({
        'full_name': 'first',
        'date': 'min',
        'injury_type': 'first',
        'injury_detail': 'first',
    })
    .rename(columns={
        'full_name': 'player_name',
        'date': 'first_report_date',
    })
)

injury_events = injury_event_base.merge(
    best_by_injury[['injury_id', 'game_id', 'play_id', 'confidence_level', 'confidence_score']],
    on='injury_id',
    how='left'
)
injury_events['loggedInGame'] = injury_events['game_id'].notna().astype(int)

# leave game_id/play_id/confidence blank when unmatched
injury_events.loc[injury_events['loggedInGame'] == 0, ['game_id', 'confidence_level', 'confidence_score']] = [None, None, None]
injury_events['play_id'] = pd.to_numeric(injury_events['play_id'], errors='coerce').astype('Int64')

injury_events.to_csv('F_injury_events.csv', index=False)

# Add linkage fields back onto full injury reports table
event_link_cols = injury_events[['injury_id', 'loggedInGame', 'game_id', 'play_id', 'confidence_level', 'confidence_score']]
report_with_logged = report.merge(event_link_cols, on='injury_id', how='left')
report_with_logged['loggedInGame'] = report_with_logged['loggedInGame'].fillna(0).astype(int)
report_with_logged['play_id'] = pd.to_numeric(report_with_logged['play_id'], errors='coerce').astype('Int64')
report_with_logged.to_csv('F_nfl_injuries_cleaned_with_logged_id.csv', index=False)

print(f"\nMatching Summary:")
print(f"Total logged injuries: {len(logged)}")
print(f"With valid next game: {len(logged_with_window)}")
print(f"Matched to injury reports: {len(best_matches)}")
print(f"Overall match rate: {len(best_matches) / len(logged) * 100:.1f}%")
print(f"Match rate (of those with next game): {len(best_matches) / len(logged_with_window) * 100:.1f}%")
print(f"Unique injury events table rows: {len(injury_events)}")
print(f"Injury events linked to in-game logs: {injury_events['loggedInGame'].sum()}")

Filled missing logged team via home/away fallback for 145 rows
Dropping 11 logged rows with unparsable names
Logged injuries with valid next game: 14436 / 14531
Strict merge rows: 252502 | with report date: 251400
Fallback merge rows: 2504 | with report date: 2264
Total candidate rows before date window: 255006
Candidates with report date: 253664
Dropped (missing report date): 1342
Dropped (report before game date): 121191
Dropped (report after next game date): 124268
Candidates after date filtering: 8205

Matching Summary:
Total logged injuries: 14531
With valid next game: 14436
Matched to injury reports: 7585
Overall match rate: 52.2%
Match rate (of those with next game): 52.5%
Unique injury events table rows: 51279
Injury events linked to in-game logs: 6741


: 